In [55]:
import mne
import numpy as np
import matplotlib.pyplot as plt
import glob
import os


In [56]:

DATA_DIR_A = "SetA"
DATA_DIR_B = "SetB"
DATA_DIR_C = "SetC"
DATA_DIR_D = "SetD"
DATA_DIR_E = "SetE"

LABEL_C1 = 0 #A
LABEL_C2 = 0 #B
LABEL_C3 = 0 #C
LABEL_C4 = 0 #D
LABEL_C5 = 1 #E

In [57]:


def load_dataset(folder_path, label):
    data = []
    labels = []

    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)

        # each file is 1 signal that is 4097 samples in Bonn dataset (23.6*Fs)
        signal = np.loadtxt(file_path)

        data.append(signal)
        labels.append(label)

    return np.array(data), np.array(labels)



X_A, Y_A = load_dataset(DATA_DIR_A, LABEL_C1)
X_B, Y_B = load_dataset(DATA_DIR_B, LABEL_C2)
X_C, Y_C = load_dataset(DATA_DIR_C, LABEL_C3)
X_D, Y_D = load_dataset(DATA_DIR_D, LABEL_C4)
X_E, Y_E = load_dataset(DATA_DIR_E, LABEL_C5)


X = np.concatenate([X_A, X_B, X_C, X_D, X_E], axis=0)
Y= np.concatenate([Y_A, Y_B, Y_C, Y_D, Y_E], axis=0)


In [58]:
import pywt

# Shannon entropy
def compute_entropy(signal):
    hist, _ = np.histogram(signal, bins=100, density=True)
    hist = hist + 1e-6  # avoid log(0)
    return -np.sum(hist * np.log2(hist))


def extract_dwt_features(signal, wavelet='db6', level=5):
    coeffs = pywt.wavedec(signal, wavelet, level=level)

    features = []


    for c in coeffs:
        features.append(np.mean(c))
        features.append(np.std(c))
        features.append(np.sum(c**2))  # energy
        features.append(compute_entropy(c))

    return np.array(features)

In [59]:
# Case A-E
X_AE = np.concatenate([X_A, X_E], axis=0)
Y_AE = np.concatenate([
    np.zeros(len(X_A)),   # Class 0 → A (non-seizure)
    np.ones(len(X_E))     # Class 1 → E (seizure)
])

X_AE_features = np.array([extract_dwt_features(x) for x in X_AE])

# X_AE.shape → (samples, features)
# Y_AE.shape → (samples,)



# Case B-E
X_BE = np.concatenate([X_B, X_E], axis=0)
Y_BE = np.concatenate([
    np.zeros(len(X_B)),   # Class 0 → B (non-seizure)
    np.ones(len(X_E))     # Class 1 → E (seizure)
])

X_BE_features = np.array([extract_dwt_features(x) for x in X_BE])

# X_BE.shape → (samples, features)
# Y_BE.shape → (samples,)



# Case C-E
X_CE = np.concatenate([X_C, X_E], axis=0)
Y_CE = np.concatenate([
    np.zeros(len(X_C)),   # Class 0 → C (non-seizure)
    np.ones(len(X_E))     # Class 1 → E (seizure)
])

X_CE_features = np.array([extract_dwt_features(x) for x in X_CE])

# X_CE.shape → (samples, features)
# Y_CE.shape → (samples,)




# Case D-E
X_DE = np.concatenate([X_D, X_E], axis=0)
Y_DE = np.concatenate([
    np.zeros(len(X_D)),   # Class 0 → D (non-seizure)
    np.ones(len(X_E))     # Class 1 → E (seizure)
])

X_DE_features = np.array([extract_dwt_features(x) for x in X_DE])

# X_DE.shape → (samples, features)
# Y_DE.shape → (samples,)




# Case ABCD-E
X_ABCD = np.concatenate([X_A, X_B, X_C, X_D], axis=0)

X_ABCDE = np.concatenate([X_A, X_B, X_C, X_D, X_E], axis=0)
Y_ABCDE = np.concatenate([
    np.zeros(len(X_ABCD)),   # Class 0 → ABCD (non-seizure)
    np.ones(len(X_E))                           # Class 1 → E (seizure)
])

X_ABCDE_features = np.array([extract_dwt_features(x) for x in X_ABCDE])

# X_ABCDE.shape → (samples, features)
# Y_ABCDE.shape → (samples,)

In [60]:
# Naive Bayes classifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix
from sklearn.naive_bayes import GaussianNB

def run_10fold_cv(M, N):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    acc_list = []

    for train_idx, test_idx in skf.split(M, N):
        X_train, X_test = M[train_idx], M[test_idx]
        Y_train, Y_test = N[train_idx], N[test_idx]

        model = GaussianNB()
        model.fit(X_train, Y_train)

        Y_pred = model.predict(X_test)

        tn, fp, fn, tp = confusion_matrix(Y_test, Y_pred).ravel()
        acc = (tp + tn) / (tp + tn + fp + fn)
        acc_list.append(acc)

    return np.mean(acc_list)

In [47]:
#SVM (test)

from sklearn.svm import SVC

def run_10fold_cv_svm(M, N):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    acc_list = []

    for train_idx, test_idx in skf.split(M, N):
        X_train, X_test = M[train_idx], M[test_idx]
        Y_train, Y_test = N[train_idx], N[test_idx]

        model = SVC(kernel='rbf')
        model.fit(X_train, Y_train)

        Y_pred = model.predict(X_test)

        tn, fp, fn, tp = confusion_matrix(Y_test, Y_pred).ravel()
        acc = (tp + tn) / (tp + tn + fp + fn)
        acc_list.append(acc)

    return np.mean(acc_list)

In [61]:
acc_AE = run_10fold_cv(X_AE_features, Y_AE)
acc_BE = run_10fold_cv(X_BE_features, Y_BE)
acc_CE = run_10fold_cv(X_CE_features, Y_CE)
acc_DE = run_10fold_cv(X_DE_features, Y_DE)
acc_ABCDE = run_10fold_cv(X_ABCDE_features, Y_ABCDE)

In [62]:
print("Accuracy(A-E):", acc_AE* 100)
print("Accuracy(B-E):", acc_BE* 100)
print("Accuracy(C-E):", acc_CE* 100)
print("Accuracy(D-E):", acc_DE* 100)
print("Accuracy(ABCD-E):", acc_ABCDE* 100)

Accuracy(A-E): 100.0
Accuracy(B-E): 99.49999999999999
Accuracy(C-E): 98.5
Accuracy(D-E): 89.99999999999999
Accuracy(ABCD-E): 95.8
